In [2]:
!pip install -q --upgrade --force-reinstall pillow==11.3.0
!pip install -q transformers gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 32.5 MB/s eta 0:00:00


In [3]:

import torch
import gradio as gr
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

# -------------------------------
# 1. Load the image captioning model
# -------------------------------
MODEL_NAME = "Salesforce/blip-image-captioning-base"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on: {device}")

processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

print("Model loaded successfully!")


# -------------------------------
# 2. Image description function
# -------------------------------
def generate_caption(image, style, max_length):
    if image is None:
        return "", "⚠️ Please upload an image first."

    try:
        image = image.convert("RGB")

        # Prepare image for BLIP
        inputs = processor(
            images=image,
            return_tensors="pt"
        ).to(device)

        # Generate caption
        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=int(max_length),
                num_beams=5,
                do_sample=False
            )

        caption = processor.decode(
            output[0],
            skip_special_tokens=True
        ).strip()

        # Format output according to selected style
        if style == "Detailed":
            result = (
                f"Image Description:\n{caption}\n\n"
                "Note: This model generates a concise caption. "
                "It may not identify every detail in the image."
            )
        elif style == "Simple":
            result = caption
        elif style == "Creative":
            result = f"✨ Visual Story\n\n{caption.capitalize()}."
        else:
            result = caption

        return result, "✅ Description generated successfully!"

    except Exception as e:
        return "", f"❌ Error: {str(e)}"


# -------------------------------
# 3. Clear function
# -------------------------------
def clear_all():
    return None, "", "Upload an image to get started."


# -------------------------------
# 4. Custom gradient UI
# -------------------------------
css = """
.gradio-container {
    max-width: 1100px !important;
    margin: auto !important;
    background: linear-gradient(135deg, #f5f3ff, #eff6ff);
}

#hero {
    text-align: center;
    padding: 28px 15px;
    border-radius: 22px;
    background: linear-gradient(120deg, #6d28d9, #2563eb);
    color: white;
    margin-bottom: 18px;
}

#hero h1 {
    font-size: 38px;
    font-weight: 800;
    margin-bottom: 8px;
}

#hero p {
    font-size: 16px;
    color: #f3f4f6;
}

.panel {
    background: white;
    border: 1px solid #e0e7ff;
    border-radius: 18px;
    padding: 16px;
}

#generate {
    background: linear-gradient(90deg, #7c3aed, #2563eb);
    color: white;
    border: none;
    font-weight: bold;
}

#generate:hover {
    filter: brightness(1.08);
}

#result textarea {
    background: #fafaff;
    border-radius: 12px;
    font-size: 16px;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="violet",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    css=css,
    title="Nova Vision AI"
) as app:

    gr.HTML("""
        <div id="hero">
            <h1>✨ Nova Vision AI</h1>
            <p>Turn your images into meaningful descriptions with AI.</p>
        </div>
    """)

    with gr.Row():
        # Left side: image upload and settings
        with gr.Column(scale=1, elem_classes=["panel"]):
            gr.Markdown("### 📷 Upload Your Image")

            image_input = gr.Image(
                type="pil",
                sources=["upload"],
                label="Choose an image",
                height=320
            )

            style_input = gr.Radio(
                choices=["Natural", "Detailed", "Simple", "Creative"],
                value="Natural",
                label="Description Style"
            )

            length_input = gr.Slider(
                minimum=20,
                maximum=100,
                value=50,
                step=10,
                label="Maximum New Tokens"
            )

            with gr.Row():
                generate_btn = gr.Button(
                    "✨ Generate Description",
                    variant="primary",
                    elem_id="generate"
                )

                clear_btn = gr.Button("🗑️ Clear")

        # Right side: generated result
        with gr.Column(scale=1, elem_classes=["panel"]):
            gr.Markdown("### 📝 Image Description")

            output_text = gr.Textbox(
                label="AI Generated Text",
                placeholder="Your image description will appear here...",
                lines=9,
                interactive=False,
                elem_id="result"
            )

            status_text = gr.Textbox(
                value="Upload an image to get started.",
                label="Status",
                interactive=False,
                lines=1
            )

            gr.Markdown(
                "💡 **Tip:** Use a clear image with a visible subject "
                "for better captioning results."
            )

    gr.Markdown(
        "---\n"
        "<center>Powered by Hugging Face Transformers · BLIP</center>"
    )

    generate_btn.click(
        fn=generate_caption,
        inputs=[image_input, style_input, length_input],
        outputs=[output_text, status_text]
    )

    image_input.change(
        fn=lambda img: (
            "", "Image uploaded. Click Generate Description."
        ) if img is not None else ("", "Upload an image to get started."),
        inputs=image_input,
        outputs=[output_text, status_text]
    )

    clear_btn.click(
        fn=clear_all,
        inputs=[],
        outputs=[image_input, output_text, status_text]
    )

# -------------------------------
# 5. Launch the application
# -------------------------------
app.launch(share=True)


Loading model on: cpu


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Model loaded successfully!


/tmp/ipykernel_1583/3601428013.py:134: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4756a2b4bc1ccf7dbe.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
